## 📚 Diccionario de Datos

| # | Variable | Tipo | Descripción |
|---|----------|------|-------------|
| 1 | `Hours_Studied` | Numérica | Horas de estudio por semana |
| 2 | `Attendance` | Numérica | Porcentaje de asistencia a clases |
| 3 | `Parental_Involvement` | Categórica | Nivel de involucramiento de los padres (`Low`, `Medium`, `High`) |
| 4 | `Access_to_Resources` | Categórica | Acceso a recursos educativos (`Low`, `Medium`, `High`) |
| 5 | `Extracurricular_Activities` | Categórica | Participación en actividades extracurriculares (`Yes`, `No`) |
| 6 | `Sleep_Hours` | Numérica | Promedio de horas de sueño por noche |
| 7 | `Previous_Scores` | Numérica | Puntaje obtenido en exámenes anteriores |
| 8 | `Motivation_Level` | Categórica | Nivel de motivación del estudiante (`Low`, `Medium`, `High`) |
| 9 | `Internet_Access` | Categórica | Acceso a internet (`Yes`, `No`) |
| 10 | `Tutoring_Sessions` | Numérica | Número de sesiones de tutoría por mes |
| 11 | `Family_Income` | Categórica | Nivel de ingresos familiares (`Low`, `Medium`, `High`) |
| 12 | `Teacher_Quality` | Categórica | Calidad del docente (`Low`, `Medium`, `High`) |
| 13 | `School_Type` | Categórica | Tipo de institución educativa (`Public`, `Private`) |
| 14 | `Peer_Influence` | Categórica | Influencia del entorno social (`Positive`, `Neutral`, `Negative`) |
| 15 | `Physical_Activity` | Numérica | Horas de actividad física por semana |
| 16 | `Learning_Disabilities` | Categórica | Presencia de discapacidades de aprendizaje (`Yes`, `No`) |
| 17 | `Parental_Education_Level` | Categórica | Nivel educativo de los padres (`High School`, `College`, `Postgraduate`) |
| 18 | `Distance_from_Home` | Categórica | Distancia del hogar a la escuela (`Near`, `Moderate`, `Far`) |
| 19 | `Gender` | Categórica | Género del estudiante (`Male`, `Female`) |
| 20 | `Exam_Score` | Numérica | 🎯 **Variable objetivo** — Puntaje obtenido en el examen final |

In [1]:
import duckdb
import os
os.listdir('../data/student_performance')
con = duckdb.connect()
con.execute("CREATE TABLE student_performance AS SELECT * FROM read_csv_auto('../data/student_performance/student_performance.csv')")
result = con.execute("SELECT * FROM student_performance").fetchdf()
result.head()


,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,Low,High,False,7,73,Low,True,0,Low,Medium,Public,Positive,3,False,High School,Near,Male,67
1,19,64,Low,Medium,False,8,59,Low,True,2,Medium,Medium,Public,Negative,4,False,College,Moderate,Female,61
2,24,98,Medium,Medium,True,7,91,Medium,True,2,Medium,Medium,Public,Neutral,4,False,Postgraduate,Near,Male,74
3,29,89,Low,Medium,True,8,98,Medium,True,1,Medium,Medium,Public,Negative,4,False,High School,Moderate,Male,71
4,19,92,Medium,Medium,True,6,65,Medium,True,3,Medium,High,Public,Neutral,4,False,College,Near,Female,70


## 🎯 Pregunta de negocio:

**¿Existe brecha de rendimiento entre estudiantes con altos vs bajos recursos académicos y acceso a internet?**

---

## 🧠 Qué estamos midiendo

Queremos comparar:

* Estudiantes con:

  * `Access_to_Resources = High`
  * `Internet_Access = Yes`

VS

* Estudiantes con:

  * `Access_to_Resources = Low`
  * `Internet_Access = No`

Y ver su:

> 🎯 Promedio de `Exam_Score`

---

In [2]:
con.execute("""
-- =====================================================
-- PREGUNTA 1: Brecha digital y de recursos académicos
-- =====================================================

SELECT
    Access_to_Resources,
    Internet_Access,
    COUNT(*) AS total_students,
    ROUND(AVG(Exam_Score), 2) AS avg_exam_score
FROM student_performance
WHERE Access_to_Resources IN ('High', 'Low')
  AND Internet_Access IN ('Yes', 'No')
GROUP BY Access_to_Resources, Internet_Access
ORDER BY avg_exam_score DESC;

-- ¿Por qué es útil para negocio educativo?
-- • Permite medir desigualdad académica causada por acceso tecnológico.
-- • Justifica inversión en infraestructura digital escolar.
-- • Identifica grupos vulnerables que necesitan apoyo prioritario.

""").fetchdf()

,Access_to_Resources,Internet_Access,total_students,avg_exam_score
0,High,True,1819,68.19
1,High,False,156,66.99
2,Low,True,1224,66.29
3,Low,False,89,65.06


## 🎯 Pregunta de negocio:

**¿Cuánto influye la cantidad de horas de estudio en el puntaje del examen?**

---

## 🧠 Qué estamos midiendo

Queremos comparar:

* Estudiantes según su nivel de horas de estudio:

  * `Bajo`  (< 10 horas)
  * `Medio` (10–19 horas)
  * `Alto`  (20+ horas)

Y analizar su:

> 🎯 Promedio de `Exam_Score`

---

In [3]:
con.execute("""
-- =====================================================
-- PREGUNTA 2: Impacto de horas de estudio en rendimiento
-- =====================================================

SELECT
    CASE
        WHEN Hours_Studied < 10 THEN 'Bajo'
        WHEN Hours_Studied < 20 THEN 'Medio'
        ELSE 'Alto'
    END AS study_level,
    COUNT(*) AS total_students,
    ROUND(AVG(Exam_Score), 2) AS avg_exam_score
FROM student_performance
GROUP BY study_level
ORDER BY avg_exam_score DESC;

-- ¿Por qué es útil para negocio educativo?
-- • Permite identificar hábitos de estudio efectivos.
-- • Ayuda a diseñar planes de refuerzo académico.
-- • Orienta recomendaciones personalizadas según dedicación académica.

""").fetchdf()

,study_level,total_students,avg_exam_score
0,Alto,3528,68.49
1,Medio,2808,65.99
2,Bajo,271,63.82


## 🎯 Pregunta de negocio:

**¿El nivel de apoyo de los padres influye en el puntaje promedio de los estudiantes?**

---

## 🧠 Qué estamos midiendo

Queremos comparar estudiantes según:

* Nivel de `Parental_Involvement`:

  * `High`
  * `Medium`
  * `Low`

Y analizar su:

> 🎯 Promedio de `Exam_Score`

---

In [4]:
con.execute("""
-- =====================================================
-- PREGUNTA 3: Influencia del apoyo parental
-- =====================================================

SELECT
    Parental_Involvement,
    COUNT(*) AS total_students,
    ROUND(AVG(Exam_Score), 2) AS avg_exam_score
FROM student_performance
GROUP BY Parental_Involvement
ORDER BY avg_exam_score DESC;

-- ¿Por qué es útil para negocio educativo?
-- • Permite evaluar el impacto del entorno familiar en el rendimiento.
-- • Ayuda a diseñar programas de acompañamiento para padres.
-- • Sustenta estrategias de orientación familiar en instituciones educativas.

""").fetchdf()

,Parental_Involvement,total_students,avg_exam_score
0,High,1908,68.09
1,Medium,3362,67.10
2,Low,1337,66.36


## 🎯 Pregunta de negocio:

**¿Existe diferencia significativa en el puntaje promedio entre estudiantes con y sin acceso a internet?**

---

## 🧠 Qué estamos midiendo

Queremos comparar:

* Estudiantes con:

  * `Internet_Access = Yes`
  * `Internet_Access = No`

Y analizar su:

> 🎯 Promedio de `Exam_Score`

---

In [5]:
con.execute("""
-- =====================================================
-- PREGUNTA 4: Influencia del acceso a internet
-- =====================================================

SELECT
    Internet_Access,
    COUNT(*) AS total_students,
    ROUND(AVG(Exam_Score), 2) AS avg_exam_score
FROM student_performance
GROUP BY Internet_Access
ORDER BY avg_exam_score DESC;

-- ¿Por qué es útil para negocio educativo?
-- • Justifica inversión en conectividad y acceso digital.
-- • Evidencia el impacto de recursos online en el desempeño.
-- • Permite reducir brechas tecnológicas entre estudiantes.

""").fetchdf()

,Internet_Access,total_students,avg_exam_score
0,True,6108,67.29
1,False,499,66.54


## 🎯 Pregunta de negocio:

**¿Cómo impacta el nivel de asistencia a clases en el puntaje promedio de los estudiantes?**

---

## 🧠 Qué estamos midiendo

Queremos comparar estudiantes según su nivel de asistencia:

* `Baja`   (< 60%)
* `Media`  (60% – 84%)
* `Alta`   (85%+)

Y analizar su:

> 🎯 Promedio de `Exam_Score`

---

In [6]:
con.execute("""
-- =====================================================
-- PREGUNTA 5: Impacto de la asistencia en el rendimiento
-- =====================================================

SELECT
    CASE
        WHEN Attendance < 60 THEN 'Baja'
        WHEN Attendance < 85 THEN 'Media'
        ELSE 'Alta'
    END AS attendance_level,
    COUNT(*) AS total_students,
    ROUND(AVG(Exam_Score), 2) AS avg_exam_score
FROM student_performance
GROUP BY attendance_level
ORDER BY avg_exam_score DESC;

-- ¿Por qué es útil para negocio educativo?
-- • Permite identificar si la presencia en clase influye directamente en el rendimiento.
-- • Ayuda a justificar políticas contra el ausentismo escolar.
-- • Facilita la detección temprana de estudiantes en riesgo académico.

""").fetchdf()

,attendance_level,total_students,avg_exam_score
0,Alta,2511,69.69
1,Media,4096,65.73


## 🎯 Pregunta de negocio:

**¿Dormir más horas mejora el rendimiento promedio de los estudiantes?*

---

## 🧠 Qué estamos midiendo

El sueño esta relacionado directamente con la capacidad de concentracion en clase, consolidacion de la memoria y el rendimiento conginitvo en evaluacion

> 🎯 Promedio de `Exam_Score`

---

In [7]:
con.execute("""
-- =====================================================
-- PREGUNTA 6: Impacto de horas de sueño en rendimiento
-- =====================================================

SELECT
    CASE
        WHEN Sleep_Hours < 6 THEN 'Pocas'
        WHEN Sleep_Hours < 8 THEN 'Adecuadas'
        ELSE 'Óptimas'
    END AS sleep_level,
    COUNT(*) AS total_students,
    ROUND(AVG(Exam_Score), 2) AS avg_exam_score
FROM student_performance
GROUP BY sleep_level
ORDER BY avg_exam_score DESC;

-- ¿Por qué es útil para negocio educativo?
-- • Relaciona hábitos de salud con desempeño académico.
-- • Permite diseñar campañas de bienestar estudiantil.
-- • Ayuda a prevenir bajo rendimiento por fatiga o mala concentración.

""").fetchdf()

,sleep_level,total_students,avg_exam_score
0,Pocas,1004,67.40
1,Adecuadas,3117,67.22
2,Óptimas,2486,67.19


## 🎯 Pregunta de negocio

**¿Los estudiantes que asisten a tutorías obtienen mejores puntajes que quienes no reciben apoyo académico adicional?**

---

## 🧠 Qué estamos midiendo

El sueño esta relacionado directamente con la capacidad de concentracion en clase, consolidacion de la memoria y el rendimiento conginitvo en evaluacion

> 🎯 Promedio de `Exam_Score`

---

In [8]:
con.execute("""
-- =====================================================
-- PREGUNTA 7: Impacto de tutorías académicas
-- =====================================================

SELECT
    Tutoring_Sessions,
    COUNT(*) AS total_students,
    ROUND(AVG(Exam_Score), 2) AS avg_exam_score
FROM student_performance
GROUP BY Tutoring_Sessions
ORDER BY avg_exam_score DESC;

-- ¿Por qué es útil para negocio educativo?
-- • Permite evaluar la efectividad de programas de refuerzo escolar.
-- • Ayuda a justificar inversión en tutorías académicas.
-- • Identifica estudiantes que más se benefician del acompañamiento.

""").fetch_df()

,Tutoring_Sessions,total_students,avg_exam_score
0,6,18,71.67
1,7,7,69.86
2,5,103,69.07
3,8,1,69.00
4,4,301,68.23
5,3,836,67.89
6,2,1649,67.57
7,1,2179,66.98
8,0,1513,66.49
